In [ ]:
import pandas as pd
import numpy as np

from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler


# =========================================================
# 1. LOAD AND CLEAN FORECAST OUTPUT DATA
# =========================================================

df = pd.read_csv("Capacity Prediction at co Level.csv")

# Clean column names
df.columns = df.columns.str.strip()

# Rename columns
df = df.rename(columns={
    "Origin": "ORIGIN_BRANCH",
    "Destination": "DESTIN_BRANCH",
    "Week": "WEEK",
    "Observed Chargeable Weight": "ACTUAL",
    "Predicted Chargeable Weight considering Holidays": "PREDICTED_HOLIDAYS",
    "Predicted Chargeable Weight": "PREDICTED_BASE"
})

# Parse date
df["WEEK"] = pd.to_datetime(df["WEEK"], format="%m/%d/%y", errors="coerce")

# Convert numeric columns
for col in ["ACTUAL", "PREDICTED_HOLIDAYS", "PREDICTED_BASE"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Clean text columns
for col in ["ORIGIN_BRANCH", "DESTIN_BRANCH"]:
    df[col] = df[col].astype(str).str.strip().str.upper()

# Drop bad rows
df = df.dropna(subset=[
    "ORIGIN_BRANCH",
    "DESTIN_BRANCH",
    "WEEK",
    "ACTUAL",
    "PREDICTED_HOLIDAYS",
    "PREDICTED_BASE"
]).copy()

# Keep only nonnegative values
df = df[
    (df["ACTUAL"] >= 0) &
    (df["PREDICTED_HOLIDAYS"] >= 0) &
    (df["PREDICTED_BASE"] >= 0)
].copy()

# Create lane
df["LANE"] = df["ORIGIN_BRANCH"] + " -> " + df["DESTIN_BRANCH"]

# Use holiday-aware prediction as main forecast
df["PREDICTED"] = df["PREDICTED_HOLIDAYS"]

# Sort
df = df.sort_values(["LANE", "WEEK"]).reset_index(drop=True)

print("Initial cleaned shape:", df.shape)
print(df.head())

In [ ]:
# =========================================================
# 2. FILTER BAD / NON-INFORMATIVE FORECAST ROWS
# =========================================================

model_base = df.copy()

# Keep only meaningful forecast rows
model_base = model_base[model_base["PREDICTED"] > 500].copy()

# Remove rows with zero actual volume
model_base = model_base[model_base["ACTUAL"] > 0].copy()

# Optional extra activity filter:
# keep rows where either actual or predicted volume is meaningfully large
model_base = model_base[
    (model_base["ACTUAL"] > 500) | (model_base["PREDICTED"] > 500)
].copy()

# Sort before lane filtering
model_base = model_base.sort_values(["LANE", "WEEK"]).reset_index(drop=True)

# -----------------------------
# FILTER LANES WITH SUFFICIENT HISTORY
# -----------------------------
lane_counts = model_base.groupby("LANE")["WEEK"].count()
valid_lanes = lane_counts[lane_counts >= 12].index

model_base = model_base[model_base["LANE"].isin(valid_lanes)].copy()

# Final sort/reset
model_base = model_base.sort_values(["LANE", "WEEK"]).reset_index(drop=True)

print("\nAfter filtering shape:", model_base.shape)
print("\nRemaining lanes:", model_base["LANE"].nunique())
print("\nLane counts:")
print(model_base["LANE"].value_counts().head(20))
print("\nPreview:")
print(model_base[["LANE", "WEEK", "ACTUAL", "PREDICTED"]].head(10))

In [ ]:
# =========================================================
# 3. RESIDUAL-BASED FEATURE ENGINEERING
# =========================================================

model_base["RESIDUAL"] = model_base["ACTUAL"] - model_base["PREDICTED"]
model_base["ABS_ERROR"] = model_base["RESIDUAL"].abs()

model_base["PCT_ERROR"] = np.where(
    model_base["PREDICTED"].abs() > 1e-6,
    model_base["RESIDUAL"] / model_base["PREDICTED"],
    0
)

model_base["ABS_PCT_ERROR"] = model_base["PCT_ERROR"].abs()

print("\nResidual feature preview:")
print(model_base[
    ["LANE", "WEEK", "ACTUAL", "PREDICTED", "RESIDUAL", "ABS_ERROR", "PCT_ERROR"]
].head(10))

In [ ]:
# =========================================================
# 4. ROLLING LANE-LEVEL BEHAVIOR FEATURES
# =========================================================

model_base = model_base.sort_values(["LANE", "WEEK"]).copy()
grp = model_base.groupby("LANE")

model_base["ROLLING_RESIDUAL_MEAN_4"] = grp["RESIDUAL"].transform(
    lambda s: s.rolling(window=4, min_periods=2).mean()
)

model_base["ROLLING_RESIDUAL_STD_4"] = grp["RESIDUAL"].transform(
    lambda s: s.rolling(window=4, min_periods=2).std()
)

model_base["ROLLING_ABS_ERROR_MEAN_4"] = grp["ABS_ERROR"].transform(
    lambda s: s.rolling(window=4, min_periods=2).mean()
)

model_base["ROLLING_PCT_ERROR_MEAN_4"] = grp["ABS_PCT_ERROR"].transform(
    lambda s: s.rolling(window=4, min_periods=2).mean()
)

model_base["ROLLING_ACTUAL_MEAN_4"] = grp["ACTUAL"].transform(
    lambda s: s.rolling(window=4, min_periods=2).mean()
)

model_base["ROLLING_PREDICTED_MEAN_4"] = grp["PREDICTED"].transform(
    lambda s: s.rolling(window=4, min_periods=2).mean()
)

model_base["ROLLING_RESIDUAL_STD_4"] = model_base["ROLLING_RESIDUAL_STD_4"].fillna(0)

print("\nRolling feature preview:")
print(model_base.head(10))


In [ ]:
# =========================================================
# 5. BUILD MODEL TABLE
# =========================================================

feature_cols = [
    "ACTUAL",
    "PREDICTED",
    "RESIDUAL",
    "ABS_ERROR",
    "PCT_ERROR",
    "ABS_PCT_ERROR",
    "ROLLING_RESIDUAL_MEAN_4",
    "ROLLING_RESIDUAL_STD_4",
    "ROLLING_ABS_ERROR_MEAN_4",
    "ROLLING_PCT_ERROR_MEAN_4",
    "ROLLING_ACTUAL_MEAN_4",
    "ROLLING_PREDICTED_MEAN_4"
]

model_df = (
    model_base
    .replace([np.inf, -np.inf], np.nan)
    .dropna(subset=feature_cols)
    .copy()
)

print("\nModel table shape:", model_df.shape)
print(model_df[feature_cols].head())

In [ ]:
# =========================================================
# 6. SCALE FEATURES
# =========================================================

X = model_df[feature_cols]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [ ]:
# =========================================================
# 7. TRAIN ISOLATION FOREST
# =========================================================

iso = IsolationForest(
    n_estimators=200,
    contamination=0.03,
    random_state=42
)

model_df["ANOMALY_LABEL"] = iso.fit_predict(X_scaled)     # -1 = anomaly, 1 = normal
model_df["ANOMALY_FLAG"] = (model_df["ANOMALY_LABEL"] == -1).astype(int)
model_df["ANOMALY_SCORE"] = iso.decision_function(X_scaled)

print("\nAnomaly counts:")
print(model_df["ANOMALY_FLAG"].value_counts())

In [ ]:
# =========================================================
# 8. CLASSIFY ANOMALY TYPE
# =========================================================

def classify_anomaly(row):
    if row["ANOMALY_FLAG"] == 0:
        return "Normal"
    elif row["RESIDUAL"] > 0:
        return "Demand Spike"
    elif row["RESIDUAL"] < 0:
        return "Demand Drop"
    else:
        return "Other"

model_df["ANOMALY_TYPE"] = model_df.apply(classify_anomaly, axis=1)



In [ ]:
# =========================================================
# 9. FINAL RESULTS TABLE
# =========================================================

results = model_df[
    [
        "ORIGIN_BRANCH",
        "DESTIN_BRANCH",
        "LANE",
        "WEEK",
        "ACTUAL",
        "PREDICTED",
        "RESIDUAL",
        "ABS_ERROR",
        "PCT_ERROR",
        "ABS_PCT_ERROR",
        "ROLLING_RESIDUAL_MEAN_4",
        "ROLLING_RESIDUAL_STD_4",
        "ANOMALY_SCORE",
        "ANOMALY_FLAG",
        "ANOMALY_TYPE"
    ]
].sort_values(["ANOMALY_FLAG", "ANOMALY_SCORE"], ascending=[False, True])

print("\nTop anomaly results:")
print(results.head(25))


In [ ]:
# =========================================================
# 10. EXTRACT TOP ANOMALIES ONLY
# =========================================================

top_anomalies = results[results["ANOMALY_FLAG"] == 1].copy()
top_anomalies = top_anomalies.sort_values("ANOMALY_SCORE")

print("\nTop flagged anomalies:")
print(top_anomalies.head(25))


In [ ]:
# =========================================================
# 11. OPTIONAL: SAVE OUTPUTS
# =========================================================

results.to_csv("lane_anomaly_results.csv", index=False)
top_anomalies.to_csv("top_lane_anomalies.csv", index=False)

print("\nSaved files:")
print("- lane_anomaly_results.csv")
print("- top_lane_anomalies.csv")

In [ ]:
print("Total rows:", len(model_df))
print("Total anomalies:", model_df["ANOMALY_FLAG"].sum())
print("Anomaly %:", model_df["ANOMALY_FLAG"].mean())

print("\nTop lanes in anomalies:")
print(
    model_df[model_df["ANOMALY_FLAG"] == 1]["LANE"]
    .value_counts()
    .head(10)
)

The anomaly detection model identified that a majority of anomalies were concentrated in the PVG → LAX lane, suggesting persistent deviations between predicted and actual freight volumes. This indicates either high demand volatility or limitations in the forecasting model for this lane, making it a priority for operational investigation